In [1]:
import csv
from pathlib import Path

root = Path('/vol/ideadata/ed52egek/pycharm/syneverything/datasets/imagenet')
inputs = [
    (root / 'ImageNet_LT_train.txt', 'TRAIN'),
    (root / 'ImageNet_LT_val.txt', 'VAL'),
    (root / 'ImageNet_LT_test.txt', 'TEST'),
]

# First pass: determine number of classes
max_id = -1
for file_path, _ in inputs:
    with file_path.open('r') as fin:
        for line in fin:
            line = line.strip()
            if not line:
                continue
            try:
                _, id_part = line.rsplit(' ', 1)
                cid = int(id_part)
                if cid > max_id:
                    max_id = cid
            except ValueError:
                continue

num_classes = max_id + 1 if max_id >= 0 else 0

out_path = root.parent / 'imagenet_lt.csv'
index_counter = 0
with out_path.open('w', newline='') as f:
    writer = csv.writer(f)
    header = ['','path','id'] + [f'label_{i}' for i in range(num_classes)] + ['Split']
    writer.writerow(header)
    for file_path, split in inputs:
        with file_path.open('r') as fin:
            for line in fin:
                line = line.strip()
                if not line:
                    continue
                try:
                    path_part, id_part = line.rsplit(' ', 1)
                    cid = int(id_part)
                except ValueError:
                    continue
                row = [index_counter, path_part, index_counter]
                # one-hot vector
                one_hot = [0] * num_classes
                one_hot[cid] = 1
                row.extend(one_hot)
                row.append(split)
                writer.writerow(row)
                index_counter += 1
print(f'Wrote {index_counter} rows with {num_classes} label columns to {out_path}')

Wrote 185846 rows with 1000 label columns to /vol/ideadata/ed52egek/pycharm/syneverything/datasets/imagenet_lt.csv
